# Mini Projet Préparation de Données Images
Classification de déchets : cardboard, glass, metal, paper, plastic, trash

Objectif : construire un jeu de données images propre et homogène à partir d'images brutes hétérogènes.

## Configuration et structure du projet

In [1]:
# Question 0 - Import des librairies
import os
import shutil
import hashlib
import numpy as np
import pandas as pd
from PIL import Image, ImageStat
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [6]:
# Question 0 - Définition des chemins et création des dossiers du projet
BASE_DIR = ".."
RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
CLEANED_DIR = os.path.join(BASE_DIR, "data", "cleaned")
REPORTS_DIR = os.path.join(BASE_DIR, "reports")

CLASSES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]

for classe in CLASSES:
    os.makedirs(os.path.join(CLEANED_DIR, classe), exist_ok=True)

print("Dossiers prêts.")


Dossiers prêts.


## Partie 1 – Exploration du dataset

In [25]:
# Question 1 - Récupérer pour chaque image : nom, classe, format, mode,
# largeur, hauteur, ecart-type des pixels, nombre de canaux, taille (poids)

def explorer_image(chemin, classe):
    infos={
        "nom":os.path.basename(chemin),
        "classe":classe,
        "chemin": chemin,
        "format":None,
        "mode":None,
        "largeur":None,
        "hauteur":None,
        "ecart_type":None,
        "nb_canaux":None,
        "taille_octets":os.path.getsize(chemin),
        "corrompue":False
    }
    try:
        with Image.open(chemin) as img:
            img.verify()  # Vérifie si l'image est corrompue
        with Image.open(chemin) as img:
            infos["format"]=img.format
            infos["mode"]=img.mode
            infos["largeur"], infos["hauteur"]=img.size
            if img.mode=="L":
                infos["nb_canaux"]=1
            else:
                infos["nb_canaux"]=len(img.getbands())
            stat=ImageStat.Stat(img.convert("L"))
            infos["ecart_type"]=stat.stddev[0]
    except Exception:
            infos["corrompue"]=True
    return infos


In [26]:
lignes = []
for classe in CLASSES:
    dossier_classe = os.path.join(RAW_DIR, classe)
    if not os.path.isdir(dossier_classe):
        continue
    for nom_fichier in os.listdir(dossier_classe):
        chemin = os.path.join(dossier_classe, nom_fichier)
        if os.path.isfile(chemin):
            lignes.append(explorer_image(chemin, classe))

df = pd.DataFrame(lignes)
print("Nombre total d'images explorées :", len(df))
df.head()

Nombre total d'images explorées : 1032


,nom,classe,chemin,format,mode,largeur,hauteur,ecart_type,nb_canaux,taille_octets,corrompue
0,cardboard1.jpg,cardboard,..\data\raw\cardboard\cardboard1.jpg,JPEG,RGB,512.0,384.0,31.875892,3.0,17333,False
1,cardboard10.jpg,cardboard,..\data\raw\cardboard\cardboard10.jpg,JPEG,RGB,512.0,384.0,38.799907,3.0,21683,False
2,cardboard100.jpg,cardboard,..\data\raw\cardboard\cardboard100.jpg,JPEG,RGB,512.0,384.0,44.498372,3.0,14884,False
3,cardboard101.jpg,cardboard,..\data\raw\cardboard\cardboard101.jpg,JPEG,RGB,512.0,384.0,68.561937,3.0,14289,False
4,cardboard102.jpg,cardboard,..\data\raw\cardboard\cardboard102.jpg,JPEG,RGB,512.0,384.0,45.320633,3.0,18015,False


## Partie 2 – Détecter les images corrompues

In [27]:
# Question 2 - Fonction de détection d'image corrompue
def est_corrompue(chemin):
    try:
        with Image.open(chemin) as img:
            img.verify()
        return False
    except Exception:
        return True

In [28]:
df_corrompues=df[df["corrompue"]==True]
print("Nombre d'images corrompues :", len(df_corrompues))
df_corrompues[["nom", "classe"]]

Nombre d'images corrompues : 6


,nom,classe
147,cardboard83.jpg,cardboard
326,glass74.jpg,glass
446,metal48.jpg,metal
633,paper213.jpg,paper
791,plastic13.jpg,plastic
1004,trash3.jpg,trash


## Partie 3 – Détecter les images vides

In [29]:
# Question 3 - Fonction de détection d'image vide (noire, blanche ou peu de variation)
SEUIL_VARIATION = 5.0

def est_vide(chemin, seuil=SEUIL_VARIATION):
    try:
        with Image.open(chemin) as img:
            stat = ImageStat.Stat(img.convert("L"))
            moyenne = stat.mean[0]
            ecart_type = stat.stddev[0]
            if ecart_type < seuil:
                return True
            if moyenne < 5 or moyenne > 250:
                return True
            return False
    except Exception:
        return False

df_non_corrompues = df[df["corrompue"] == False].copy()
df_non_corrompues["vide"] = df_non_corrompues["chemin"].apply(est_vide)

df_vides = df_non_corrompues[df_non_corrompues["vide"] == True]
print("Nombre d'images quasi vides :", len(df_vides))
df_vides[["nom", "classe"]]

Nombre d'images quasi vides : 4


,nom,classe
167,image-blanche-512x384.jpg,cardboard
355,image-noire-512x384.png,glass
357,image-blanche-512x384.jpg,metal
358,image-noire-512x384.png,metal


## Partie 4 – Détecter les différences de résolution

In [31]:
# Question 4.1 - Résolution min, max, les plus fréquentes et nombre d'images par résolution

df_valides=df_non_corrompues[df_non_corrompues["vide"]==False].copy()
df_valides["resolution"]=df_valides["largeur"].astype(str) + "x" + df_valides["hauteur"].astype(str)

resolution_min=(df_valides["largeur"].min(), df_valides["hauteur"].min())
resolution_max=(df_valides["largeur"].max(), df_valides["hauteur"].max())
compte_resolutions=df_valides["resolution"].value_counts()

print("Résolution minimale :", resolution_min)
print("Résolution maximale :", resolution_max)
print("\nRésolutions les plus fréquentes :")
compte_resolutions.head(10)


Résolution minimale : (np.float64(32.0), np.float64(32.0))
Résolution maximale : (np.float64(512.0), np.float64(384.0))

Résolutions les plus fréquentes :


resolution
512.0x384.0    1009
32.0x32.0         5
48.0x32.0         4
40.0x40.0         4
Name: count, dtype: int64

In [32]:
# Question 4.2 - Images ne respectant pas la contrainte 64x64 minimum
TAILLE_MIN=64

df_trop_petites=df_valides[(df_valides["largeur"] < TAILLE_MIN) | (df_valides["hauteur"] < TAILLE_MIN)]
print("Nombre d'images trop petites (< 64x64) :", len(df_trop_petites))
df_trop_petites[["nom", "classe", "largeur", "hauteur"]]

Nombre d'images trop petites (< 64x64) : 13


,nom,classe,largeur,hauteur
20,cardboard117.jpg,cardboard,48.0,32.0
77,cardboard22.jpg,cardboard,32.0,32.0
133,cardboard70.jpg,cardboard,40.0,40.0
171,glass100.jpg,glass,40.0,40.0
229,glass15.jpg,glass,48.0,32.0
268,glass21.jpg,glass,32.0,32.0
270,glass23.jpg,glass,32.0,32.0
383,metal121.jpg,metal,48.0,32.0
413,metal2.jpg,metal,32.0,32.0
421,metal26.jpg,metal,40.0,40.0
